# Анализ: цены, ассортимент и рейтинги в двух учебных магазинах

**Исследовательский вопрос.** Как соотносятся уровень цен и структура ассортимента двух учебных
интернет-магазинов — `scrapingcourse.com/ecommerce` (HTML) и `DummyJSON` (REST API) — в сопоставимых
товарных группах, и связаны ли цена и скидка с рейтингом товара и оценками в отзывах?

**Связь источников:** единые товарные группы (`product_groups`): HTML-товар относится к группе по
категориям карточки (запасной вариант — по названию), API-товар — по slug категории DummyJSON
(правила в `config.yaml → linkage`). Вторая сущность — отзывы DummyJSON (`reviews`, связь N:1 с `products`).

> Ноутбук читает только БД и `mart/`, сеть не нужна. Сначала выполните `python -m src.run_pipeline`,
> затем *Run All*. Графики сохраняются в `mart/fig_*.png`.

In [ ]:
import json, os, sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

BASE = Path(os.environ.get("PIPELINE_BASE", "."))   # корень проекта (для mock: mock_run)
DB = BASE / "db" / "warehouse.db"
MART = BASE / "mart"
REPORTS = BASE / "reports"
MART.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 160)

con = sqlite3.connect(DB)
products = pd.read_sql_query("SELECT * FROM products", con)
reviews = pd.read_sql_query("SELECT * FROM reviews", con)
html = products[products.source == "scrapingcourse.com"]
api = products[products.source == "dummyjson.com"]
print(f"Товаров: {len(products)} (HTML: {len(html)}, API: {len(api)}); отзывов: {len(reviews)}")
pd.read_sql_query("SELECT * FROM v_source_overview", con)

## 0. Паспорт прогона: метрики сбора и качества

In [ ]:
run = json.loads((REPORTS / "run_report_latest.json").read_text(encoding="utf-8"))
q_path = sorted(REPORTS.glob("quality_*.json"))[-1]
quality = json.loads(q_path.read_text(encoding="utf-8"))
print(f"Прогон {run['run_id']}: {run['duration_sec']} c, запросов {run['requests_total']}, коды {run['status_codes']}, "
      f"304: {run['not_modified_304']}, ошибок {run['error_rate']:.1%}, p50 {run['latency_median_ms']} мс, "
      f"p95 {run['latency_p95_ms']} мс, трафик {run['traffic_bytes_wire']/1024:.0f} КБ")
print("Вердикт качества:", quality["status"], quality["reasons"])
pd.DataFrame(quality["metrics"])[["name", "value", "direction", "warn", "critical", "level"]]

## 1. Цены в сопоставимых товарных группах

Сравниваем только группы, представленные в обоих источниках (`v_overlap_groups`). Распределения цен
скошены, поэтому смотрим медианы и критерий Манна — Уитни (непараметрический), а ось цен — логарифмическая.

In [ ]:
overlap = pd.read_sql_query("SELECT * FROM v_overlap_groups ORDER BY product_group", con)
groups = overlap.product_group.tolist()
both = products[products.product_group.isin(groups)].copy()
both["Источник"] = both.source.map({"scrapingcourse.com": "HTML: scrapingcourse", "dummyjson.com": "API: DummyJSON"})

rows = []
for g in groups:
    a = both[(both.product_group == g) & (both.source == "scrapingcourse.com")].price_usd
    b = both[(both.product_group == g) & (both.source == "dummyjson.com")].price_usd
    p = stats.mannwhitneyu(a, b).pvalue if len(a) > 2 and len(b) > 2 else np.nan
    rows.append({"группа": g, "n_html": len(a), "n_api": len(b), "медиана_html": round(a.median(), 2),
                 "медиана_api": round(b.median(), 2), "отношение_медиан_api/html": round(b.median() / a.median(), 2),
                 "p_value (Mann-Whitney)": round(p, 4)})
price_cmp = pd.DataFrame(rows)
price_cmp.to_csv(MART / "price_comparison_by_group.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 5))
sns.boxplot(data=both, x="product_group", y="price_usd", hue="Источник", ax=ax, showfliers=True)
ax.set_yscale("log")
ax.set_title("Рис. 1. Цены в сопоставимых товарных группах (логарифмическая шкала)")
ax.set_xlabel("Товарная группа"); ax.set_ylabel("Цена, $")
plt.tight_layout(); plt.savefig(MART / "fig_1_price_by_group.png", dpi=120); plt.show()
price_cmp

## 2. Структура ассортимента: доли товарных групп в каждом магазине

In [ ]:
mix = (products.groupby(["source", "product_group"]).size()
       .groupby(level=0).apply(lambda s: s / s.sum() * 100).unstack(fill_value=0).round(1))
mix.index = mix.index.get_level_values(-1)
ax = mix.plot(kind="barh", stacked=True, figsize=(11, 4), colormap="tab20")
ax.set_title("Рис. 2. Структура ассортимента по товарным группам, % товаров")
ax.set_xlabel("% товаров источника"); ax.set_ylabel("")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout(); plt.savefig(MART / "fig_2_assortment_mix.png", dpi=120); plt.show()
stock = products.groupby("source").agg(товаров=("business_key", "count"),
                                        в_наличии_pct=("in_stock", lambda s: round(s.mean() * 100, 1)),
                                        со_скидкой_pct=("discount_pct", lambda s: round((s > 0).mean() * 100, 1)),
                                        средняя_скидка_pct=("discount_pct", lambda s: round(s.mean(), 2)))
display(mix)
stock

## 3. Связаны ли цена и скидка с рейтингом? (DummyJSON)

Рейтинг товара есть только в API. Используем ранговую корреляцию Спирмена (связь может быть нелинейной).

In [ ]:
rho_price, p_price = stats.spearmanr(api.price_usd, api.rating)
rho_disc, p_disc = stats.spearmanr(api.discount_pct, api.rating)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
sc = axes[0].scatter(api.price_usd, api.rating, c=api.discount_pct, cmap="viridis", alpha=0.75)
axes[0].set_xscale("log"); axes[0].set_xlabel("Цена, $ (лог.)"); axes[0].set_ylabel("Рейтинг товара")
axes[0].set_title(f"Цена и рейтинг: Spearman ρ = {rho_price:.2f} (p = {p_price:.3f})")
plt.colorbar(sc, ax=axes[0], label="Скидка, %")
band = pd.read_sql_query("SELECT * FROM v_price_rating ORDER BY price_band", con)
axes[1].bar(band.price_band, band.avg_rating, color="steelblue")
for i, (r, n) in enumerate(zip(band.avg_rating, band.n_products)):
    axes[1].text(i, r + 0.02, f"n={n}", ha="center", fontsize=8)
axes[1].set_ylim(band.avg_rating.min() - 0.3, 5); axes[1].set_title("Средний рейтинг по ценовым диапазонам")
axes[1].tick_params(axis="x", rotation=20)
fig.suptitle("Рис. 3. DummyJSON: цена, скидка и рейтинг")
plt.tight_layout(); plt.savefig(MART / "fig_3_price_vs_rating.png", dpi=120); plt.show()
print(f"Скидка и рейтинг: Spearman ρ = {rho_disc:.2f} (p = {p_disc:.3f})")
band

## 4. Согласуется ли рейтинг товара с оценками в отзывах? (две связанные сущности)

`products.rating` — витринный рейтинг, `reviews.rating` — оценки конкретных отзывов (3 на товар в DummyJSON).

In [ ]:
cons = pd.read_sql_query("SELECT * FROM v_review_consistency WHERE n_reviews > 0", con)
rho_rev, p_rev = stats.spearmanr(cons.product_rating, cons.avg_review_rating)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(cons.product_rating, cons.avg_review_rating, alpha=0.6)
axes[0].plot([1, 5], [1, 5], "r--", lw=1, label="y = x")
axes[0].set_xlabel("Рейтинг товара (products)"); axes[0].set_ylabel("Средняя оценка отзывов (reviews)")
axes[0].set_title(f"Spearman ρ = {rho_rev:.2f} (p = {p_rev:.3f})"); axes[0].legend()
sns.histplot(cons.rating_gap, bins=20, ax=axes[1], color="darkorange")
axes[1].set_title("Разница: рейтинг товара − средняя оценка отзывов"); axes[1].set_xlabel("Разница, баллов")
fig.suptitle("Рис. 4. Согласованность витринного рейтинга и отзывов")
plt.tight_layout(); plt.savefig(MART / "fig_4_rating_vs_reviews.png", dpi=120); plt.show()
cons[["product_rating", "avg_review_rating", "rating_gap"]].describe().round(2)

## 5. Динамика по прогонам (журнал `runs`, история цен `product_history`)

In [ ]:
runs = pd.read_sql_query("SELECT * FROM runs ORDER BY started_at", con)
hist = pd.read_sql_query("SELECT business_key, COUNT(*) AS versions FROM product_history GROUP BY 1", con)
print(f"Прогонов в журнале: {len(runs)}; товаров с изменением цены/наличия: {(hist.versions > 1).sum()}")
if len(runs) >= 2:
    fig, ax1 = plt.subplots(figsize=(11, 4))
    x = range(len(runs))
    ax1.bar(x, runs.requests_total, color="lightgray", label="HTTP-запросов")
    ax2 = ax1.twinx(); ax2.plot(x, runs.products_clean, "o-", color="tab:blue", label="товаров в clean")
    ax1.set_xticks(list(x)); ax1.set_xticklabels([r[:15] for r in runs.run_id], rotation=30, fontsize=7)
    ax1.set_ylabel("Запросов"); ax2.set_ylabel("Товаров")
    ax1.set_title("Рис. 5. Прогоны: число запросов (инкрементальность) и объём чистого слоя")
    fig.legend(loc="upper right"); plt.tight_layout(); plt.savefig(MART / "fig_5_runs.png", dpi=120); plt.show()
runs[["run_id", "mode", "quality_status", "products_clean", "reviews_clean", "inserted", "updated",
      "unchanged", "requests_total", "error_rate", "exit_code"]]

## 6. Выводы (формируются из данных текущего прогона)

In [ ]:
def fmt_p(p):
    return "статистически значимо (p < 0.05)" if p < 0.05 else f"статистически незначимо (p = {p:.2f})"

def strength(rho):
    a = abs(rho)
    return "практически отсутствует" if a < 0.1 else "слабая" if a < 0.3 else "умеренная" if a < 0.6 else "сильная"

conclusions = []
# Вывод 1 — уровень цен в сопоставимых группах
r = price_cmp.sort_values("отношение_медиан_api/html")
col = "отношение_медиан_api/html"
fmt = lambda d: ", ".join(f"{g} ×{x}" for g, x in zip(d["группа"], d[col]))
cheaper, similar = r[r[col] < 0.9], r[(r[col] >= 0.9) & (r[col] <= 1.1)]
pricier, extreme = r[(r[col] > 1.1) & (r[col] <= 10)], r[r[col] > 10]
n_sig = int((price_cmp["p_value (Mann-Whitney)"] < 0.05).sum())
parts = []
if len(cheaper): parts.append(f"DummyJSON дешевле — {fmt(cheaper)}")
if len(similar): parts.append(f"цены сопоставимы — {fmt(similar)}")
if len(pricier): parts.append(f"DummyJSON дороже — {fmt(pricier)}")
if len(extreme): parts.append(f"другой ценовой сегмент — {fmt(extreme)} (медиана API в {extreme[col].max():.0f}+ раз выше)")
conclusions.append(
    f"1) Уровень цен. Единой картины нет — соотношение медианных цен (API / HTML) зависит от группы: "
    + "; ".join(parts) + f". Различие статистически значимо в {n_sig} из {len(price_cmp)} групп "
    f"(критерий Манна — Уитни, p < 0.05); выборки внутри групп малые (от {int(min(price_cmp.n_html.min(), price_cmp.n_api.min()))} товаров).")
# Вывод 2 — структура ассортимента
top_html = mix.loc["scrapingcourse.com"].sort_values(ascending=False).head(2)
other_api = mix.loc["dummyjson.com"].get("other", 0)
conclusions.append(
    f"2) Ассортимент. scrapingcourse — специализированный магазин одежды и спортивных товаров "
    f"(крупнейшие группы: {', '.join(f'{k} {v}%' for k, v in top_html.items())}); у DummyJSON {other_api}% "
    f"товаров приходится на категории, которых нет во втором магазине (электроника, косметика, продукты и т.п.). "
    f"Пересечение ассортимента — групп: {len(groups)} ({', '.join(groups)}).")
# Вывод 3 — рейтинг
conclusions.append(
    f"3) Рейтинг. Связь цены и рейтинга {strength(rho_price)} (ρ = {rho_price:.2f}, {fmt_p(p_price)}), "
    f"связь скидки и рейтинга — {strength(rho_disc)} (ρ = {rho_disc:.2f}, {fmt_p(p_disc)}). "
    f"Связь витринного рейтинга со средней оценкой отзывов — {strength(rho_rev)} "
    f"(ρ = {rho_rev:.2f}); средняя разница {cons.rating_gap.mean():+.2f} балла"
    + (" — по 3 отзывам на товар витринный рейтинг не восстанавливается." if abs(rho_rev) < 0.3 else "."))
print("\n\n".join(conclusions))
Path(MART / "conclusions.txt").write_text("\n\n".join(conclusions), encoding="utf-8")

## 7. Ограничения: чего данные не показывают

* **Учебные (синтетические) источники.** Оба магазина созданы для тренировки сбора данных: цены,
  рейтинги и отзывы не отражают реальный рынок. Выводы описывают *эти каталоги*, а не e-commerce в целом.
* **Охват.** scrapingcourse — ~190 товаров одежды/спорта, DummyJSON — ~190 товаров 24 категорий;
  пересекаются только несколько групп, внутри групп — единицы—десятки товаров (малые выборки, широкие
  доверительные интервалы).
* **Сопоставление по правилам.** Товарная группа HTML-товара определяется ключевыми словами по категориям и
  названию — возможны ошибки классификации (контролируются метрикой `html_unmapped_group_pct`).
* **Нет рейтингов в HTML.** Вопрос о рейтинге проверяется только на DummyJSON; для scrapingcourse
  отзывов/рейтингов в каталоге нет (или почти нет).
* **Отзывы.** В DummyJSON по 3 отзыва на товар, без ПДн (имя и e-mail авторов не сохраняем) — нельзя
  анализировать авторов и повторные покупки.
* **Период сбора.** Каталоги статичны; динамика цен (`product_history`) появится только при изменениях на
  стороне источника. Валюта — USD в обоих источниках, налоги/доставка не учитываются.
* **Пропуски.** Доля пропусков и отбраковки — в отчёте качества (раздел 0) и `clean/rejects.jsonl`.